<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
print("Ready. Month =", MONTH)

Ready. Month = 2026-03


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** score each page by how far its click-through rate
sits below what pages at the same search position typically get, weighted by how
much impression volume backs that gap up. A big CTR shortfall on a high-traffic
page ranks higher than the same shortfall on a barely-seen page.

### score = (tier_expected_ctr - page_ctr) * log1p(impressions)

**Reason code:** `ctr_below_tier_expected` — the page underperforms its own
position tier's average CTR, not some universal number (comparing pages across
different tiers directly would be misleading, since position 1 always gets more
clicks than position 20 regardless of quality).

**Action label:** `review_metadata_or_snippet` — the standard lever for a CTR
gap at a decent position; the page is being seen and ranked, it's just not being
clicked as much as position-peers.

Below are the two signal checks that justify building the rule this way.

##  signal check 1 (CTR vs position tier, flag-linked to CTR-fix logic):

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

ctr_by_tier = con.sql(f"""
    WITH page_month AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT
        CASE
            WHEN avg_position <= 3 THEN '1-3'
            WHEN avg_position <= 10 THEN '4-10'
            WHEN avg_position <= 20 THEN '11-20'
            WHEN avg_position <= 50 THEN '21-50'
            ELSE '51+'
        END AS position_tier,
        COUNT(*) AS n,
        SUM(clicks)::FLOAT / NULLIF(SUM(impressions), 0) AS avg_ctr
    FROM page_month
    GROUP BY position_tier
    ORDER BY MIN(avg_position)
""").df()
print("Signal check 1 — CTR by position tier (behind FlyRank's CTR-fix logic):")
ctr_by_tier

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal check 1 — CTR by position tier (behind FlyRank's CTR-fix logic):


,position_tier,n,avg_ctr
0,1-3,17578,0.004064
1,4-10,81988,0.003239
2,11-20,32203,0.003052
3,21-50,33288,0.001366
4,51+,11681,0.000392


## signal check 2 (impression volume, behind quick-win logic):

In [3]:
volume_by_tier = con.sql(f"""
    WITH page_month AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               AVG(gsc_avg_position) AS avg_position
        FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) > 0
    )
    SELECT
        CASE
            WHEN impressions < 50 THEN 'under_50'
            WHEN impressions < 500 THEN '50_to_500'
            WHEN impressions < 5000 THEN '500_to_5000'
            ELSE '5000_plus'
        END AS impression_tier,
        COUNT(*) AS n,
        AVG(avg_position) AS avg_position,
        SUM(clicks)::FLOAT / NULLIF(SUM(impressions), 0) AS avg_ctr
    FROM page_month
    GROUP BY impression_tier
    ORDER BY MIN(impressions)
""").df()
print("Signal check 2 — CTR/position by impression volume (behind quick-win logic):")
volume_by_tier

Signal check 2 — CTR/position by impression volume (behind quick-win logic):


,impression_tier,n,avg_position,avg_ctr
0,under_50,60624,16.957822,0.004587
1,50_to_500,54190,19.950272,0.002322
2,500_to_5000,48632,11.656512,0.002973
3,5000_plus,13292,11.408703,0.002936


**Signal 1 verdict: CONFIRMED.** CTR decreases monotonically as position tier
worsens — 1-3 has the highest avg_ctr (0.0041), dropping through 4-10 (0.0032),
11-20 (0.0031), 21-50 (0.0014), down to 51+ (0.0004), roughly a 10x drop from
best to worst tier. Every tier has a large n (11,681 to 81,988), so this isn't
noise. This confirms the signal behind FlyRank's CTR-fix logic: comparing a
page's CTR only against its own position tier is the right approach, since raw
CTR is almost entirely explained by position alone.

**Signal 2 verdict: MIXED.** Impression volume does NOT cleanly predict CTR the
way position does — CTR is actually highest in the lowest-volume tier
(under_50: 0.0046), dips in 50_to_500 (0.0023), then recovers slightly in
500_to_5000 and 5000_plus (~0.0029-0.0030). This isn't monotonic, and it's
confounded by position: under_50 has the worst average position (16.96) yet the
highest CTR, and 5000_plus has the best average position (11.41) but only
middling CTR. This suggests volume alone is a weak/misleading signal for CTR —
the earlier position-tier signal is doing the real work, and volume is better
used only as a confidence weight (as the rule does via log1p(impressions)), not
as a standalone ranking signal. A clearly negative result here is still useful:
it justifies NOT building a volume-only rule and confirms position-tier
normalization is the signal that actually matters.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import numpy as np

page_month = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 50
""").df()

page_month["ctr"] = page_month["clicks"] / page_month["impressions"]

def tier(pos):
    if pos <= 3: return "1-3"
    elif pos <= 10: return "4-10"
    elif pos <= 20: return "11-20"
    elif pos <= 50: return "21-50"
    else: return "51+"

page_month["position_tier"] = page_month["avg_position"].apply(tier)
page_month["tier_expected_ctr"] = page_month.groupby("position_tier")["ctr"].transform("mean")

page_month["baseline_action_score"] = (
    (page_month["tier_expected_ctr"] - page_month["ctr"]) * np.log1p(page_month["impressions"])
)
page_month["reason_code"] = "ctr_below_tier_expected"
page_month["action"] = "review_metadata_or_snippet"

queue = page_month.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} ranked rows to work/outputs/baseline_action_score.csv")
queue.head(20)


Wrote 116114 ranked rows to work/outputs/baseline_action_score.csv


,content_hash_id,impressions,clicks,avg_position,ctr,position_tier,tier_expected_ctr,baseline_action_score,reason_code,action
0,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,4-10,0.003293,0.039010,ctr_below_tier_expected,review_metadata_or_snippet
1,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,4-10,0.003293,0.038815,ctr_below_tier_expected,review_metadata_or_snippet
2,content_d61fc394d10cba41,38000.0,1.0,2.740744,0.000026,1-3,0.003682,0.038547,ctr_below_tier_expected,review_metadata_or_snippet
3,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,4-10,0.003293,0.038531,ctr_below_tier_expected,review_metadata_or_snippet
4,content_fc67675904376267,60172.0,18.0,2.261303,0.000299,1-3,0.003682,0.037225,ctr_below_tier_expected,review_metadata_or_snippet
5,content_cd3d932d4e1c8db0,89332.0,4.0,7.786219,0.000045,4-10,0.003293,0.037033,ctr_below_tier_expected,review_metadata_or_snippet
6,content_66bf45eb0c5bb550,24259.0,1.0,2.784944,0.000041,1-3,0.003682,0.036756,ctr_below_tier_expected,review_metadata_or_snippet
7,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,1-3,0.003682,0.036710,ctr_below_tier_expected,review_metadata_or_snippet
8,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,4-10,0.003293,0.036540,ctr_below_tier_expected,review_metadata_or_snippet
9,content_046fc480045b88f5,83788.0,6.0,7.289152,0.000072,4-10,0.003293,0.036521,ctr_below_tier_expected,review_metadata_or_snippet


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Every row in this top 20 shares the same shape: very high impression volume,
good position (mostly single digits, all under 10), and near-zero CTR — several
rows have 0-1 clicks against tens or hundreds of thousands of impressions.
Reviewing by pattern group rather than fully independently, since the reasoning
repeats:

1. **content_44f34c0a90047651** — 212,404 impr, 24 clicks, pos 7.3, ctr 0.0001.
   Action: review_metadata_or_snippet. Confidence: high — huge volume backs the gap.
   Wrong if: this is a tracking/attribution artifact rather than a real snippet problem.
2. **content_8e1334d6356668e3** — 134,984 impr, 1 click, pos 4.5, ctr 0.000007.
   Confidence: high on volume, but 1 click at position 4.5 is unusually extreme —
   wrong if this reflects a data anomaly (e.g. a bot-inflated impression count)
   rather than a genuine snippet failure.
3. **content_d61fc394d10cba41** — 38,000 impr, 1 click, pos 2.7, ctr 0.00003.
   Position 2.7 with essentially zero clicks is the most suspicious combination
   in the list. Wrong if: impressions are inflated/misattributed rather than real
   searcher exposure.
4. **content_fec55986a1868d62** — 124,075 impr, 1 click, pos 9.4. Same profile
   as #2 — high volume, near-zero engagement. Wrong if data anomaly, not content issue.
5. **content_fc67675904376267** — 60,172 impr, 18 clicks, pos 2.3, ctr 0.0003.
   Best position in the top 20 with still very low CTR — a strong, more
   plausible metadata-review candidate since the click count is non-trivial (18,
   not 0-1). Wrong if: SERP feature (featured snippet, PAA) above it is
   absorbing clicks despite high position.
6. **content_cd3d932d4e1c8db0** — 89,332 impr, 4 clicks, pos 7.8. Wrong if:
   anomalous impression count.
7. **content_66bf45eb0c5bb550** — 24,259 impr, 1 click, pos 2.8. Same pattern as #3.
8. **content_306bc78dff1eb683** — 80,821 impr, 35 clicks, pos 1.5, ctr 0.0004.
   Best position of the entire top 20 (1.5!) with the most clicks (35) among
   near-zero-CTR rows — genuinely the most credible metadata/snippet candidate
   here, since there's enough click signal to trust the gap is real. Wrong if:
   a non-clickable SERP feature (e.g. this page IS the featured snippet source)
   is suppressing normal clicks.
9. **content_f6116743b00afc2d** — 107,584 impr, 15 clicks, pos 9.5. Wrong if
   data anomaly.
10. **content_046fc480045b88f5** — 83,788 impr, 6 clicks, pos 7.3. Same pattern.
11. **content_425715547c6a3ea8** — 71,513 impr, 3 clicks, pos 6.4. Same pattern.
12. **content_b9acd1ebff7d34ff** — 25,941 impr, 3 clicks, pos 2.4. Same pattern as #3.
13. **content_9540d884af3e41fd** — 82,376 impr, 11 clicks, pos 7.8. Same pattern.
14. **content_34a70fea29d15f24** — 143,019 impr, 43 clicks, pos 3.2, ctr 0.0003.
    Highest click count in the whole top 20 (43) at a strong position (3.2) —
    along with #8, this is one of the more credible genuine snippet-review
    candidates, since 43 clicks is real signal, not noise.
15. **content_1d7764b642f7bb9f** — 23,402 impr, 4 clicks, pos 1.8. Excellent
    position with almost no clicks — highly suspicious, same anomaly concern as #3.
16. **content_bf078007df823490** — 44,707 impr, **0 clicks**, pos 7.9. Zero
    clicks at meaningful volume — wrong if impressions are inflated; if real,
    this is an extreme snippet failure worth manual inspection first.
17. **content_945d6ff91386c817** — 58,278 impr, 5 clicks, pos 6.1. Same pattern.
18. **content_0c5606abaaab3178** — 38,865 impr, **0 clicks**, pos 5.7. Same as #16.
19. **content_fa17add7836d36c3** — 12,588 impr, **0 clicks**, pos 1.9. Excellent
    position, zero clicks — most suspicious position/click mismatch alongside #3 and #15.
20. **content_37a6fac676c8cebb** — 48,049 impr, 4 clicks, pos 4.4. Same pattern.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20)[["content_hash_id", "impressions", "clicks", "avg_position",
                          "ctr", "tier_expected_ctr", "baseline_action_score",
                          "reason_code", "action"]]
top20


,content_hash_id,impressions,clicks,avg_position,ctr,tier_expected_ctr,baseline_action_score,reason_code,action
0,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,0.003293,0.039010,ctr_below_tier_expected,review_metadata_or_snippet
1,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,0.003293,0.038815,ctr_below_tier_expected,review_metadata_or_snippet
2,content_d61fc394d10cba41,38000.0,1.0,2.740744,0.000026,0.003682,0.038547,ctr_below_tier_expected,review_metadata_or_snippet
3,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,0.003293,0.038531,ctr_below_tier_expected,review_metadata_or_snippet
4,content_fc67675904376267,60172.0,18.0,2.261303,0.000299,0.003682,0.037225,ctr_below_tier_expected,review_metadata_or_snippet
5,content_cd3d932d4e1c8db0,89332.0,4.0,7.786219,0.000045,0.003293,0.037033,ctr_below_tier_expected,review_metadata_or_snippet
6,content_66bf45eb0c5bb550,24259.0,1.0,2.784944,0.000041,0.003682,0.036756,ctr_below_tier_expected,review_metadata_or_snippet
7,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,0.003682,0.036710,ctr_below_tier_expected,review_metadata_or_snippet
8,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,0.003293,0.036540,ctr_below_tier_expected,review_metadata_or_snippet
9,content_046fc480045b88f5,83788.0,6.0,7.289152,0.000072,0.003293,0.036521,ctr_below_tier_expected,review_metadata_or_snippet


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks:** rows with 0-1 clicks at very good position (#2, #3, #7, #12,
#15, #16, #18, #19) are actually the *least* trustworthy despite scoring highest —
a page ranking at position 2-4 getting literally zero or one click out of tens
of thousands of impressions is unusual enough to suggest a data or tracking
anomaly (e.g. impressions logged for a query that never really reached a human,
or an attribution mismatch) rather than a genuine title/snippet failure. The
rule's score formula can't distinguish "real snippet problem" from "anomalous
impression count" — that distinction needs a human to actually look at the page.

By contrast, rows with double-digit clicks at good position (#5, #8, #14) are
more credible candidates: enough click volume to trust the CTR gap is a real
pattern, not an artifact.

**A concrete improvement for the next iteration:** flag rows with clicks < 5
separately from rows with clicks >= 5, since the near-zero-click rows likely
need a data-quality check before a content review, not a metadata rewrite.

**Leakage check:**
- No product flags used: health_score, priority_score, action_type, and
  refresh-tier flags were never queried in this notebook.
- No future window: all features (gsc_impressions, gsc_clicks, gsc_avg_position)
  are aggregated within the same month (2026-03) being scored.
- No label-derived inputs: tier_expected_ctr is computed from the same month's
  observed CTR, not from any future or predicted outcome.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

used_columns = ["content_hash_id", "gsc_impressions", "gsc_clicks", "gsc_avg_position", "gsc_data_available"]
forbidden = ["health_score", "priority_score", "action_type", "refresh_tier"]
print(f"Columns used: {used_columns}")
print(f"Forbidden product-flag columns present: {[c for c in forbidden if c in used_columns]}")

Columns used: ['content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available']
Forbidden product-flag columns present: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.